# 1、LangSmith概述
## 1.1 什么是LangSmith?
LangSmith 是 LangChain 生态系统中专门用于 LLM（大语言模型）应用调试、监控、评估和管理 的平台。
*  追踪(tracing)：记录每次 LLM 调用的详细信息
*  监控(monitoring)：实时查看应用性能
*  调试(debug)：排查问题和优化性能
*  评估(evaluate)：系统化测试 LLM 应用


In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

print(model.invoke("你好,请介绍一下你自己"))


content='你好呀！很高兴认识你！😊\n\n我是 **DeepSeek**，一个由深度求索公司创造的AI助手。让我简单介绍一下自己：\n\n## 我的基本特点\n\n- **身份**：AI智能助手，专注于理解和生成文字内容\n- **知识截止**：2026年2月\n- **版本**：DeepSeek最新版模型\n\n## 我能做什么\n\n✅ **文本处理**：回答问题、写作、翻译、编程、分析等\n✅ **文件阅读**：支持上传图像、TXT、PDF、PPT、Word、Excel文件并提取文字信息\n✅ **图片识别**：可以接收你上传的图片，识别和分析其中的可见信息\n✅ **长文本处理**：上下文可达1M，能一次性处理《三体》三部曲这样的长篇内容\n✅ **联网搜索**：支持联网功能（需要你在Web/App手动开启）\n✅ **语音输入**：App端支持语音交互\n\n## 我的优势\n\n- 🆓 **完全免费**：没有任何收费计划\n- 📱 **多平台**：Web端和App端都可以使用\n- 🎯 **专注实用**：用热情、细腻的方式提供有帮助的回答\n\n有什么我可以帮你的吗？无论是学习、工作还是生活上的问题，随时都可以问我！' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户让我介绍一下自己。这是一个非常常见且简单的开场问题。用户可能是第一次接触我，想了解我的基本身份和能力，以便后续更好地使用我。\n\n我需要给出一个清晰、友好、全面的自我介绍。应该涵盖我的身份、核心功能、关键特点，并保持热情和乐于助人的语气。想到了可以从问候开始，然后说明我是谁、由谁创造，接着分块介绍我的主要能力，比如文本处理、文件支持、上下文长度、联网搜索和语音输入等，最后强调免费和联系方式，并以开放性问题结束，邀请用户提出具体需求。\n\n回复结构可以这样：先问候并表明身份，然后用概括性语言介绍核心能力，再以要点形式列出关键特点，最后表达服务意愿并引导对话继续。注意避免技术性过强，保持信息易懂且有帮助。'} response_metadata={'token_usage': {'completion_tokens': 440, 'prompt_tokens': 35, 'total_tokens': 47

在配置文件.env中添加以下内容：
```
LANGSMITH_API_KEY=
LANGSMITH_ENDPOINT=
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=
```
即可正常使用。在WEB页面查询AGENT运行记录。


**环境变量说明**

| 变量 | 作用 | 说明 |
|---|---|---|
| `LANGSMITH_TRACING` | 是否开启追踪 | 设为 `true` 后，LangChain 的调用会**自动**上传，不需要改代码 |
| `LANGSMITH_API_KEY` | LangSmith 的 API 密钥 | 以 `lsv2_` 开头，在 LangSmith 网页的 Settings → API Keys 中创建 |
| `LANGSMITH_ENDPOINT` | 服务地址 | 美国区为 `https://api.smith.langchain.com`，欧洲区为 `https://eu.api.smith.langchain.com`，要与注册账号时选的区域一致 |
| `LANGSMITH_PROJECT` | 项目名称 | 追踪记录按项目归类。不设置时为 `default`；项目不存在时会自动创建 |

还有几个可选变量：

| 变量 | 作用 |
|---|---|
| `LANGSMITH_WORKSPACE_ID` | 账号下有多个工作空间时，指定使用哪一个 |
| `LANGSMITH_HIDE_INPUTS` / `LANGSMITH_HIDE_OUTPUTS` | 设为 `true` 后不上传输入 / 输出内容，只记录耗时、token 等信息，用于保护隐私 |
| `LANGSMITH_TRACING_SAMPLING_RATE` | 采样率（0～1），比如 `0.1` 表示只上传 10% 的调用，用于调用量很大的生产环境 |

> ⚠️ 两个注意点：
> 1. 早期教程中的 `LANGCHAIN_TRACING_V2`、`LANGCHAIN_API_KEY` 等 `LANGCHAIN_` 开头的旧变量名仍然可以用，新项目建议统一用 `LANGSMITH_` 开头。
> 2. **LangSmith 只在第一次用到时读取这些环境变量，之后就缓存起来了**。在 Jupyter 中修改 `.env` 后，要**重启内核**才会生效；想在运行中途临时关闭追踪或换项目，要用 2.4 节的 `tracing_context()`。

配置好之后，上面第一个单元格中的调用，以及下面两个例子，都会自动出现在 LangSmith 网页的项目中。

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

messages = [
    "请用一句话介绍一下你自己",
    "中国的首都是哪里?",
    "日本的首相是谁?",
]

results = model.batch(messages, config={"max_concurrency": 2})
print(type(results))
for question, result in zip(messages, results):
    print(f"问题：{question}")
    print(f"回答：{result.text}\n")

<class 'list'>
问题：请用一句话介绍一下你自己
回答：我是 Codex，一个协助你编写、调试和改进代码的 AI 助手。

问题：中国的首都是哪里?
回答：中国的首都是北京。

问题：日本的首相是谁?
回答：截至2026年6月，日本首相是高市早苗。



In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    configurable_fields=("model", "temperature"),  # 允许在调用时修改 model 和 temperature
)
question = "请回答:当前日本的首相是谁？"

response1 = model.invoke(question)  # 使用默认的 deepseek-v4-flash
response2 = model.invoke(
    question,
    config={"configurable": {"model": "deepseek:deepseek-v4-pro"}},  # 本次调用换成 deepseek-v4-pro
)
print(f"默认模型：{response1.response_metadata['model_name']}，回答：{response1.content}")
print(f"切换后：{response2.response_metadata['model_name']}，回答：{response2.content}")

默认模型：deepseek-flash，回答：当前日本首相（内阁总理大臣）是**高市早苗**。她于2025年10月21日就任，是日本首位女性首相。
切换后：deepseek-v4-pro，回答：截至2026年4月，日本首相是 **石破茂**。


这两个例子在 LangSmith 中的样子：

- `batch()` 中的 3 个问题，各自生成一条追踪记录；
- 可配置模型的两次调用也各生成一条记录，点开能看到每次实际使用的模型。

另外注意，同一个问题两个模型的回答不一样。到底哪个对？单靠人工逐条去看效率很低，这正是第 4 节"数据集与评估"要解决的问题。

## 1.2 具体功能
功能1：核心应用与开发
1. Tracing（追踪）
    * 功能：这是 LangSmith 最核心的功能。它会完整记录你大模型应用的每一次调用链路（Trace）。
    * 作用：当你的 Agent（智能体）或 RAG 系统运行变慢或报错时，点击进入对应的项目（如上图中的 langchain1.2_smith ），你可以看到每一步具体的 Prompt 是什么、模型返回了什么、消耗了多少 Token，以及每一个链条节点的耗时，非常方便排查 Bug 和优化性能。

2. Monitoring（监控）
    * 功能：提供生产环境的高级数据可视化看板。
    * 作用：帮你从宏观角度监控应用在一段时间内的运行状况。你可以看到 Token 消耗趋势、QPS（每秒请求数）、错误率、平均延迟（Latency）以及成本预估。适合应用上线后观察系统的稳定性和开销。

3. Datasets & Experiments（数据集与实验）
    * 功能：用于管理测试数据集并运行对比实验。
    * 作用：你可以把用户的真实输入、特定的边界情况（Edge Cases）存为数据集。当你修改了Prompt 或更换了底层大模型时，可以在这里运行自动化对比测试，直观看到新旧版本在同一批测试集上的表现差异。

4. Evaluators（评估器）
    * 功能：配置和自动化评估任务。
    * 作用：大模型的输出往往难以用传统的断言（Assert）来测试。这里允许你配置基于规则（如关键词匹配）或基于模型（LLM-as-a-judge）的评估指标（如：答案相关性、是否包含幻觉等），对追踪到的数据或实验结果进行自动打分。

5. Annotation Queues（标注队列）
    * 功能：人工反馈与数据清洗工具。
    * 作用：在应用开发或初上线阶段，你可以把一部分痕迹（Traces）发送到标注队列中，让团队中的核心成员、业务专家或人工客服进行手动打分、纠正回答或贴标签，这些高质量的人工标注数据后续可直接用于微调模型或充当测试集。


功能2:提示词与调试工具

1. Prompts（提示词管理）
    * 功能：类似“提示词版的 GitHub”。
    *  作用：把 Prompt 从代码中解耦出来，统一在云端管理。你可以在这里对 Prompt 进行版本控制（如 v1 、 v2 ），直接在代码中通过 API 动态拉取最新的提示词。它还支持团队协作和 Prompt的分享。
    
2. Playground（演练场）
    * 功能：一个网页端的模型交互界面。
    * 作用：无需写任何代码，直接在这里选择不同的模型（如 OpenAI、Anthropic 或是本地模型），快速微调并测试你的 Prompt 效果，还可以一键将调整好的 Prompt 保存到上方的 Prompts 仓库中。
3. Studio（工作室）
    * 功能：通常与 LangGraph 深度集成，提供可视化的图形交互界面。
    * 作用：如果你的应用是基于图结构（Graph-based）的复杂复杂 Agent 架构，Studio 可以让你可视化地看到状态机（State）在各个节点之间的流转，甚至支持在某个节点“暂停”，手动修改数据后再继续向下执行，是调试复杂智能体交互的利器。
4. Context Hub（上下文中心）
    * 功能：管理全局上下文或通用组件配置。
    * 作用：用于存放可在多个项目或 Prompt 中复用的公共上下文模板、全局变量或系统预设提示。

功能3:部署与沙盒
1. Deployments（部署）
    * 功能：一键将你的 LangChain 应用或 LangGraph Agent 部署为线上可用的 API 服务（通常依托于LangGraph Cloud）。
    * 作用：提供开箱即用的生产端点，帮你处理高并发、队列管理和状态持久化，让你专注于编写业务逻辑。
2. Sandboxes（沙盒）
    * 功能：提供轻量级的在线运行和测试环境。
    * 作用：在不污染生产环境的前提下，供开发人员安全地试运行、测试新部署的 Agent 或执行自动化脚本。


## 1.3 其他功能

除了上面 4 个核心功能，LangSmith 还提供：

5. **Feedback（反馈）**：给某次调用打分、写评语。分数可以来自用户（👍/👎），也可以来自代码或人工审核，用来找出回答不好的案例。
6. **Annotation Queues（标注队列）**：把需要人工审核的追踪记录放进队列，由团队成员逐条查看、打分、修正，修正后的结果可以加入数据集。
7. **Prompt 管理**：在网页上保存、修改提示词并记录版本（Prompt Hub），还可以在 Playground 中直接调试提示词、对比不同模型的效果。
8. **自动化规则与在线评估**：按条件（比如出错、反馈分数低）自动处理新的追踪记录，例如把它们加入数据集或标注队列；也可以对线上的真实调用自动运行评估器打分。
9. **告警（Alerts）**：错误率、延迟等指标超过阈值时发出通知。
10. **部署（Deployment）**：把用 LangGraph 编写的 Agent 部署到 LangSmith 上运行，并提供 Studio 可视化调试界面（后续章节会学到）。

LangSmith **不依赖 LangChain**：不用 LangChain 的代码，也可以用 `@traceable` 等方式接入（见 2.2、2.3 节）。

下面按"追踪 → 查询 → 反馈 → 评估"的顺序，用代码演示这些功能：

| 功能 | 章节 | 方式 |
|---|---|---|
| Tracing（追踪） | 第 2 节 | 代码 |
| 用代码查询追踪数据 | 2.5 节 | 代码 |
| Feedback（反馈） | 第 3 节 | 代码 |
| Datasets & Experiments、Evaluators | 第 4 节 | 代码 |
| Monitoring、告警、自动化规则、标注队列、Prompt 管理 | 第 5 节 | 网页操作 |

# 2、追踪（Tracing）

## 2.1 LangChain 的调用会被自动追踪

开启追踪后，每次调用模型都会生成一条**追踪记录（Trace）**，在后台上传到 LangSmith，**不会阻塞**程序运行。一条追踪记录包含：

- 输入（发给模型的消息）和输出（模型的回答）；
- 耗时、token 用量，以及 LangSmith 按模型价格**估算的费用**；
- 模型名称、调用参数，以及我们传入的 `run_name`、`tags`、`metadata`。

`04-model-invoke.ipynb` 中讲过：LangChain 把每次调用记录为一条"运行记录（Run）"，并用 `collect_runs()` 在本地查看过。**LangSmith 保存的就是这些运行记录**，只不过它们被上传到了云端，可以在网页上查看、搜索和统计。

下面调用一次模型，并打印这条追踪记录在网页上的链接：

In [1]:
from langchain_core.tracers.context import collect_runs
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

with collect_runs() as cb:  # 在本地也收集一份运行记录，用来获取网页链接
    response = model.invoke(
        "用一句话介绍 LangSmith",
        config={"run_name": "介绍LangSmith", "tags": ["笔记演示"], "metadata": {"user_id": "u_001"}},
    )
print("回答:", response.content)

wait_for_all_tracers()  # 追踪数据在后台上传，这里等它上传完成
print("在 LangSmith 中查看:", cb.traced_runs[0].get_url())

回答: LangSmith 是 LangChain 推出的一个用于调试、测试、评估和监控 LLM 应用的全生命周期开发运维平台。


在 LangSmith 中查看: https://smith.langchain.com/o/f63305d7-b586-40bc-bf5c-8f1a53cdb7a3/projects/p/9e047f6d-e745-4c44-98b2-5c48c6149de1/trace/01a0d37c-5fd5-73d2-8900-654d71e0bfee/run/01a0d37c-5fd5-73d2-8900-654d71e0bfee?start_time=2026-09-24T12%3A55%3A38.721784%2B00%3A00


打开链接，可以看到这次调用的输入、输出、耗时、token 用量和费用，右侧的 Metadata 中有我们传入的 `user_id`，列表中可以按 `笔记演示` 标签筛选。

## 2.2 用 @traceable 追踪自己的函数

真实应用中，一次请求往往包含多个步骤：调用模型、查数据库、处理文本等。给函数加上 `@traceable` 装饰器，这个函数也会被记录下来，**函数内部的模型调用会自动成为它的子步骤**，整个流程在 LangSmith 中显示为一棵树。

`@traceable` 常用参数：

- `name`：在追踪记录中显示的名称，默认是函数名；
- `run_type`：步骤类型，默认为 `chain`，还可以是 `llm`、`tool`、`retriever` 等，网页上会用不同图标显示。

In [2]:
import uuid
from langsmith import traceable
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

@traceable(run_type="tool", name="统计字数")
def count_chars(text: str) -> int:
    return len(text)

@traceable(name="翻译并统计")  # 外层函数：包含一次模型调用和一次普通函数调用
def translate_and_count(text: str) -> dict:
    english = model.invoke(f"把下面的话翻译成英文，只输出译文：{text}").content
    return {"english": english, "chars": count_chars(english)}

translate_run_id = uuid.uuid4()  # 自己指定运行 ID，2.5 节会用它查询这条追踪
result = translate_and_count("今天天气真好", langsmith_extra={"run_id": translate_run_id})
print(result)

{'english': 'The weather is really nice today.', 'chars': 33}


在 LangSmith 中打开"翻译并统计"这条记录，可以看到它下面有两个子步骤：模型调用 `ChatDeepSeek` 和工具 `统计字数`，每一步的输入、输出和耗时都能单独查看（2.5 节会用代码把这棵树打印出来）。

## 2.3 追踪不用 LangChain 的代码

LangSmith 不依赖 LangChain。直接使用 openai SDK 时，用 `wrap_openai()` 包装一下客户端，它发出的请求就会被追踪。DeepSeek 兼容 OpenAI 接口，所以同样适用：

In [3]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

client = wrap_openai(OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url=os.getenv("DEEPSEEK_BASE_URL")))

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{"role": "user", "content": "1 + 1 等于几？只回答数字"}],
    extra_body={"thinking": {"type": "disabled"}},
)
print(response.choices[0].message.content)

2


## 2.4 控制追踪：tracing_context()

`tracing_context()` 可以临时修改 `with` 代码块内的追踪设置，常用参数：

| 参数 | 作用 |
|---|---|
| `enabled=False` | 代码块内的调用**不追踪**，比如处理敏感数据时 |
| `project_name="..."` | 代码块内的追踪记录发到**另一个项目**，比如把测试数据和正式数据分开 |
| `tags=[...]`、`metadata={...}` | 给代码块内的所有追踪记录统一加上标签、元数据 |

前面提到环境变量会被缓存，所以运行中途要改追踪设置，只能用这种方式。

In [4]:
from langsmith import tracing_context
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

with tracing_context(enabled=False):  # 这个代码块内的调用不会上传到 LangSmith
    response = model.invoke("你好，请用一句话回复")
print("回答:", response.content)

回答: 你好！很高兴见到你，请问有什么我可以帮你的吗？


## 2.5 用代码查询追踪数据

除了在网页上查看，还可以用 langsmith SDK 查询追踪数据，做统计分析（比如统计每天的 token 用量和费用）。

> ⚠️ **新版 SDK 的查询接口是异步的**：从 langsmith 0.14 起，旧的 `client.list_runs()` 已被弃用（会出现 DeprecationWarning），改用 `client.runs.query()`，要用 `async for` 遍历。Jupyter 中可以直接写 `await` 和 `async for`；在普通 `.py` 文件中，要放进 `async def` 函数，再用 `asyncio.run()` 运行（参考 06 中的异步调用写法）。
>
> 新接口的几个规则：
> - `min_start_time` 默认只查**最近 1 天**，要查更早的数据需要自己指定；
> - `selects` 指定要返回的字段，字段名用**大写**，不指定时只返回 ID；
> - `run_type` 的值也要**大写**，如 `"LLM"`、`"CHAIN"`。

下面查询最近 7 天的顶层追踪记录，打印耗时、token 用量和费用：

In [5]:
from datetime import datetime, timedelta
from langsmith import Client
from langchain_core.tracers.langchain import wait_for_all_tracers
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

wait_for_all_tracers()  # 确保前面的追踪数据都已上传
ls_client = Client()
project = await ls_client.aread_project(project_name=os.getenv("LANGSMITH_PROJECT"))  # 查询时要用项目 ID

print(f"{'时间':<12}{'名称':<16}{'耗时(秒)':>9}{'输入tokens':>12}{'输出tokens':>12}{'费用(美元)':>12}  输入内容")
async for run in ls_client.runs.query(
    project_ids=[str(project.id)],
    is_root=True,                                       # 只看最外层的记录，不看子步骤
    min_start_time=datetime.now() - timedelta(days=7),  # 默认只查最近 1 天
    page_size=20,
    selects=["NAME", "START_TIME", "LATENCY_SECONDS", "PROMPT_TOKENS", "COMPLETION_TOKENS", "TOTAL_COST", "INPUTS_PREVIEW"],
):
    start = run.start_time.astimezone().strftime("%m-%d %H:%M")
    preview = (run.inputs_preview or "").replace("\n", " ")[:24]
    print(f"{start:<12}{run.name:<16}{run.latency_seconds or 0:>9.2f}{run.prompt_tokens or 0:>12}"
          f"{run.completion_tokens or 0:>12}{run.total_cost or 0:>12.6f}  {preview}")

时间          名称                  耗时(秒)    输入tokens    输出tokens      费用(美元)  输入内容


09-24 05:55 ChatOpenAI           1.43          15           1    0.000000  user: 1 + 1 等于几？只回答数字
09-24 05:55 翻译并统计                1.48           0           0    0.000000  今天天气真好
09-24 05:55 介绍LangSmith          1.45           9          27    0.000018  human: 用一句话介绍 LangSmith
09-24 05:53 加法                   0.02           0           0    0.000000  {"a":1,"b":2}
09-24 05:52 翻译并统计                1.58           0           0    0.000000  今天天气真好
09-24 05:51 ChatDeepSeek         1.40          13           3    0.000004  human: 水的化学式是什么？只回答化学式
09-24 05:50 ChatOpenAI           1.30          12           1    0.000000  user: 1+1=? 只回答数字
09-24 05:50 ChatDeepSeek         1.55          13           1    0.000003  human: 中国的首都是哪里？只回答城市名
09-24 02:34 ChatDeepSeek         6.08          92         314    0.000313  human: 请回答:当前日本的首相是谁？
09-24 02:34 ChatDeepSeek         7.81          39        1278    0.000773  human: 请回答:当前日本的首相是谁？
09-24 02:23 ChatOpenAI           1.77        4392          19    0.0

表格说明：

- 最新的记录在最上面。表中**没有** 2.4 节那次调用，说明 `tracing_context(enabled=False)` 生效了；
- 输入以 `user:` 开头的 `ChatOpenAI` 记录，来自 2.3 节用 `wrap_openai` 直接调用的 DeepSeek（`wrap_openai` 默认把记录命名为 ChatOpenAI）；
- `翻译并统计` 这类 chain 记录本身不直接调用模型，所以 token 显示为 0，用量记在它的子步骤上；
- 费用是 LangSmith 按它的模型价格表**估算**的，价格表中匹配不到模型时显示为 0（比如 `wrap_openai` 的那条记录）。

再按 2.2 中指定的运行 ID，把"翻译并统计"这条追踪的树形结构打印出来（`trace_id` 就是最外层运行的 ID）：

In [6]:
import asyncio

await asyncio.sleep(3)  # 服务端保存数据有几秒延迟，稍等一下再查
runs = [
    run
    async for run in ls_client.runs.query(
        project_ids=[str(project.id)],
        trace_id=str(translate_run_id),
        selects=["NAME", "RUN_TYPE", "LATENCY_SECONDS", "PARENT_RUN_IDS", "DOTTED_ORDER"],
    )
]
for run in sorted(runs, key=lambda r: r.dotted_order):  # dotted_order 按执行顺序排序
    depth = len(run.parent_run_ids or [])                 # 有几个上级，就缩进几层
    print("    " * depth + f"{run.name}（{run.run_type}），耗时 {run.latency_seconds:.2f} 秒")

翻译并统计（chain），耗时 1.48 秒
    ChatDeepSeek（llm），耗时 1.48 秒
    统计字数（tool），耗时 0.00 秒


## 2.6 案例：用追踪数据发现问题

看 2.5 中打印的表格：几条 `ChatOpenAI` 记录（本笔记开头用中转服务调用的 `gpt-6-luna`），输入内容只有"日本的首相是谁?"这样一句话，**输入 token 却都在 4392 左右**；而 DeepSeek 回答类似的问题，输入只有几十个 token。

这说明中转服务在服务器端给每个请求额外加了 4000 多 token 的内容（很可能是一段系统提示），这部分同样计费。开头 batch 例子中模型自称"我是 Codex"，很可能也是这段额外内容造成的。

这类问题只看代码是发现不了的，因为代码里发送的确实只有一句话。而追踪记录把每次调用的 token 用量和费用都记了下来，**异常一目了然**。这就是 Tracing 在排查问题、控制成本上的价值。

# 3、反馈（Feedback）

反馈就是给某次调用**打分、写评语**，分数会显示在追踪记录上，可以按分数筛选。常见用法：

- 应用界面上放 👍/👎 按钮，用户点击后把分数记到对应的调用上；
- 用代码自动检查回答（比如是否包含敏感词），把结果记为反馈；
- 定期筛选出分数低的调用，分析原因，或者加入数据集用于评估。

给某次调用添加反馈，需要知道它的**运行 ID**。可以像 2.2 节那样自己指定：通过 config 的 `run_id` 键（`04-model-invoke.ipynb` 4.1.2 节的表格中提到过）。

> 新版 SDK 要求创建反馈时提供 `session_id`（即追踪记录所在项目的 ID），否则会出现弃用警告。

In [7]:
import uuid
from langsmith import Client
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

ls_client = Client()
project = ls_client.read_project(project_name=os.getenv("LANGSMITH_PROJECT"))

run_id = uuid.uuid4()  # 自己指定运行 ID，之后用它添加反馈
response = model.invoke("水的化学式是什么？", config={"run_id": run_id})
print("回答:", response.content)

wait_for_all_tracers()  # 等追踪数据上传完成，再添加反馈
feedback = ls_client.create_feedback(
    run_id=run_id,
    key="user_rating",      # 反馈的名称，可以自定义
    score=1,                # 分数，比如 1 表示👍、0 表示👎
    comment="回答正确",      # 可选的文字说明
    session_id=project.id,  # 追踪记录所在项目的 ID
)
print("已添加反馈:", feedback.key, feedback.score, feedback.comment)

回答: 水的化学式是 **H₂O**。

这表示每个水分子由 **2个氢原子** 和 **1个氧原子** 组成。


已添加反馈: user_rating 1 回答正确


在 LangSmith 中打开这次调用，右侧的 Feedback 区域可以看到 `user_rating: 1` 和评语。项目的追踪列表中也会多出 `user_rating` 一列，可以按它筛选和排序。

# 4、数据集与评估（Datasets & Evaluation）

## 4.1 为什么需要评估？

修改提示词、更换模型之后，回答变好了还是变差了？只凭感觉看几个例子是不可靠的（比如本笔记开头可配置模型的例子中，两个模型对同一个问题给出了不同的回答）。评估的做法是：

1. 准备一批有**标准答案**的问题，组成**数据集（Dataset）**，其中每个问题叫一个**样本（Example）**；
2. 用被测试的程序（**被测函数**，target）回答每个问题；
3. 用**评估器（Evaluator）**给每个回答打分；
4. 一次完整的"回答 + 打分"叫一次**实验（Experiment）**。修改提示词或模型后再跑一次实验，就能用数据对比两个版本。

## 4.2 创建数据集

数据集保存在 LangSmith 中。每个样本包含 `inputs`（输入）和 `outputs`（标准答案）。下面的代码在数据集已经存在时直接使用，避免重复运行时重复创建：

In [8]:
from langsmith import Client
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

ls_client = Client()
DATASET_NAME = "langchain-learn-常识问答"

examples = [
    {"inputs": {"question": "中国的首都是哪里？"}, "outputs": {"answer": "北京"}},
    {"inputs": {"question": "水的化学式是什么？"}, "outputs": {"answer": "H2O"}},
    {"inputs": {"question": "《红楼梦》的作者是谁？"}, "outputs": {"answer": "曹雪芹"}},
    {"inputs": {"question": "太阳系中最大的行星是哪个？"}, "outputs": {"answer": "木星"}},
    {"inputs": {"question": "一年有几个季节？"}, "outputs": {"answer": "4个"}},
]

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
    print("数据集已存在，直接使用")
else:
    dataset = ls_client.create_dataset(dataset_name=DATASET_NAME, description="LangSmith 笔记演示用的常识问答")
    ls_client.create_examples(dataset_id=dataset.id, examples=examples)
    print("已创建数据集")

print("数据集:", dataset.name, "| 样本数:", len(list(ls_client.list_examples(dataset_id=dataset.id))))

已创建数据集


数据集: langchain-learn-常识问答 | 样本数: 5


在 LangSmith 网页的 **Datasets & Experiments** 页面中可以看到这个数据集，也可以在网页上直接增删、修改样本。

## 4.3 定义被测函数和评估器

**被测函数**接收一个样本的 `inputs`，返回一个字典作为输出。这里准备两个版本，稍后对比：一个直接提问，一个加上"简洁回答"的系统提示。

**评估器**就是普通的 Python 函数，按参数名接收需要的数据：`inputs`（输入）、`outputs`（被测函数的输出）、`reference_outputs`（标准答案）。返回值可以是：

- `True` / `False` 或数字：分数，函数名就是这项指标的名称；
- 字典 `{"key": 指标名, "score": 分数, "comment": 说明}`：可以附带打分理由。

下面定义三个评估器：

| 评估器 | 类型 | 判断标准 |
|---|---|---|
| `contains_answer` | 规则 | 回答中是否包含标准答案的原文 |
| `llm_judge` | 大模型当裁判（LLM-as-a-judge） | 让另一个模型判断回答和标准答案的意思是否一致 |
| `is_concise` | 规则 | 回答是否不超过 50 个字符 |

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

# ---------- 被测函数：两个版本 ----------
def target_default(inputs: dict) -> dict:
    """版本一：直接提问"""
    return {"answer": model.invoke(inputs["question"]).content}

def target_concise(inputs: dict) -> dict:
    """版本二：加上"简洁回答"的系统提示"""
    messages = [("system", "请用一句话简洁地回答问题"), ("human", inputs["question"])]
    return {"answer": model.invoke(messages).content}

# ---------- 评估器 ----------
def contains_answer(outputs: dict, reference_outputs: dict) -> bool:
    """规则评估：回答中是否包含标准答案的原文"""
    return reference_outputs["answer"] in outputs["answer"]

def is_concise(outputs: dict) -> bool:
    """规则评估：回答是否简洁"""
    return len(outputs["answer"]) <= 50

judge = init_chat_model(  # 裁判模型，temperature=0 让判断尽量稳定
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    extra_body={"thinking": {"type": "disabled"}},
)

def llm_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """大模型当裁判：判断回答与标准答案的意思是否一致"""
    prompt = (
        f"问题：{inputs['question']}\n"
        f"标准答案：{reference_outputs['answer']}\n"
        f"待评估的回答：{outputs['answer']}\n\n"
        "待评估的回答与标准答案的意思是否一致？只回答“正确”或“错误”。"
    )
    verdict = judge.invoke(prompt).content.strip()
    return {"key": "llm_judge", "score": verdict.startswith("正确"), "comment": verdict}

def show_results(results):
    """打印一次实验中每个样本的得分，以及各项指标的平均分"""
    rows = list(results)
    totals = {}
    for row in rows:
        scores = {r.key: r.score for r in row["evaluation_results"]["results"]}
        for key, score in scores.items():
            totals[key] = totals.get(key, 0) + score
        answer = row["run"].outputs["answer"].replace("\n", " ")
        print(f"{row['example'].inputs['question']}")
        print(f"    包含原文={scores['contains_answer']!s:<6} 裁判={scores['llm_judge']!s:<6} "
              f"简洁={scores['is_concise']!s:<6} 回答（{len(answer)} 字）：{answer[:40]}")
    print("平均分：", {key: round(total / len(rows), 2) for key, total in totals.items()})
    print("实验名称：", results.experiment_name)

## 4.4 运行实验

用 `client.evaluate()` 运行实验：它会用被测函数回答数据集中的每个问题，再用每个评估器打分，结果上传到 LangSmith。常用参数：

| 参数 | 作用 |
|---|---|
| 第一个参数 | 被测函数 |
| `data` | 数据集名称 |
| `evaluators` | 评估器列表 |
| `experiment_prefix` | 实验名称的前缀，LangSmith 会在后面加上随机后缀 |
| `max_concurrency` | 最大并发数，同时评估几个样本 |

先运行版本一：

In [10]:
results_default = ls_client.evaluate(
    target_default,
    data=DATASET_NAME,
    evaluators=[contains_answer, llm_judge, is_concise],
    experiment_prefix="直接提问",
    max_concurrency=2,
)
show_results(results_default)

View the evaluation results for experiment: '直接提问-e7cbc73e' at:
https://smith.langchain.com/o/f63305d7-b586-40bc-bf5c-8f1a53cdb7a3/datasets/fb43ba88-01ab-4357-8d20-9173b19a5b9a/compare?selectedSessions=e701bc44-f5aa-42e8-af2c-855e36f886d4




太阳系中最大的行星是哪个？
    包含原文=True   裁判=True   简洁=False  回答（80 字）：太阳系中最大的行星是**木星**。  它的直径约为 14.3 万公里，是地球直径
一年有几个季节？
    包含原文=False  裁判=False  简洁=False  回答（446 字）：这是一个有点“陷阱”的常识题，答案取决于你从哪个角度理解：  **1. 通常的答
中国的首都是哪里？
    包含原文=True   裁判=True   简洁=True   回答（13 字）：中国的首都是**北京**。
《红楼梦》的作者是谁？
    包含原文=True   裁判=True   简洁=False  回答（365 字）：关于《红楼梦》的作者，目前学界普遍认为：  **前80回的作者是曹雪芹。**  
水的化学式是什么？
    包含原文=False  裁判=True   简洁=False  回答（54 字）：水的化学式是 **H₂O**。  这表示一个水分子由 **2 个氢原子** 和 
平均分： {'contains_answer': 0.6, 'llm_judge': 0.8, 'is_concise': 0.2}
实验名称： 直接提问-e7cbc73e


## 4.5 修改提示词后再跑一次，对比两个版本

In [11]:
results_concise = ls_client.evaluate(
    target_concise,
    data=DATASET_NAME,
    evaluators=[contains_answer, llm_judge, is_concise],
    experiment_prefix="简洁提示词",
    max_concurrency=2,
)
show_results(results_concise)
print("\n在网页上对比两次实验：", results_concise.comparison_url)

View the evaluation results for experiment: '简洁提示词-fa50fe8b' at:
https://smith.langchain.com/o/f63305d7-b586-40bc-bf5c-8f1a53cdb7a3/datasets/fb43ba88-01ab-4357-8d20-9173b19a5b9a/compare?selectedSessions=1d5d3b7a-f58d-4944-93af-a6821b9cb36b




太阳系中最大的行星是哪个？
    包含原文=True   裁判=True   简洁=True   回答（3 字）：木星。
一年有几个季节？
    包含原文=False  裁判=True   简洁=True   回答（8 字）：一年有四个季节。
《红楼梦》的作者是谁？
    包含原文=True   裁判=True   简洁=True   回答（13 字）：《红楼梦》的作者是曹雪芹。
中国的首都是哪里？
    包含原文=True   裁判=True   简洁=True   回答（9 字）：中国的首都是北京。
水的化学式是什么？
    包含原文=False  裁判=True   简洁=True   回答（3 字）：H₂O
平均分： {'contains_answer': 0.6, 'llm_judge': 1.0, 'is_concise': 1.0}
实验名称： 简洁提示词-fa50fe8b

在网页上对比两次实验： https://smith.langchain.com/o/f63305d7-b586-40bc-bf5c-8f1a53cdb7a3/datasets/fb43ba88-01ab-4357-8d20-9173b19a5b9a/compare?selectedSessions=1d5d3b7a-f58d-4944-93af-a6821b9cb36b


**实验结果分析**

| 指标 | 直接提问 | 简洁提示词 |
|---|:---:|:---:|
| contains_answer（包含原文） | 0.6 | 0.6 |
| llm_judge（大模型裁判） | 0.8 | 1.0 |
| is_concise（简洁） | 0.2 | 1.0 |

从中可以看出：

1. **规则评估器容易误判**：模型回答"H₂O"（2 是下标）、"四个季节"，意思完全正确，但与标准答案"H2O""4个"的写法不同，`contains_answer` 就判为错误；大模型裁判能理解意思，判断更准确。所以规则评估适合检查格式、长度、关键词等**明确的要求**，判断"意思对不对"要用 LLM-as-a-judge。
2. **提示词的效果可以量化**：加上"简洁回答"的系统提示后，简洁度从 0.2 提高到 1.0。不加提示时，"一年有几个季节"的回答长达 446 个字符，列举了两季、四季、六季等多种说法，裁判认为与标准答案不一致，判为错误。
3. **裁判也需要调教**：这里的裁判只回答"正确 / 错误"，看不到判断理由。实际使用时，可以要求裁判先说明理由再下结论，并把理由放进 `comment`，方便排查。
4. 打印顺序与数据集中的顺序不同，因为 `max_concurrency=2` 时多个样本是同时评估的。

在 LangSmith 网页上打开这个数据集，实验列表中有这两次实验。在列表中同时选中它们，就能进入对比视图，逐条并排查看两个版本的回答和分数。

# 5、网页端功能

以下功能主要在 LangSmith 网页上操作，不需要写代码：

| 功能 | 在哪里 | 用途 |
|---|---|---|
| **Monitoring（监控）** | 项目页面的监控 / Dashboards | 查看一段时间内的调用量、错误率、延迟、token 用量和费用趋势，也可以自定义看板 |
| **Alerts（告警）** | 项目的告警设置 | 错误率、延迟等指标超过阈值时发通知 |
| **Automations（自动化规则）** | 项目的 Rules 设置 | 按条件（如出错、反馈分数低、带某个标签）自动处理新的追踪记录：加入数据集、加入标注队列，或触发 Webhook |
| **Online Evaluators（在线评估）** | 项目的评估器设置 | 对线上的真实调用自动运行评估器（如 LLM-as-a-judge）打分，不需要数据集和标准答案 |
| **Annotation Queues（标注队列）** | 左侧菜单 | 团队成员逐条人工审核追踪记录、打分、修正答案 |
| **Prompts / Playground** | 左侧菜单 | 保存提示词并管理版本；在 Playground 中直接修改提示词、切换模型、查看效果 |

网页上的菜单名称和位置可能随版本调整，以实际页面为准。

# 6、注意事项与总结

**注意事项**

1. **隐私**：开启追踪后，所有输入输出都会上传到 LangSmith 的服务器。处理敏感数据时，用 `tracing_context(enabled=False)` 关闭追踪，或者设置 `LANGSMITH_HIDE_INPUTS` / `LANGSMITH_HIDE_OUTPUTS`。
2. **环境变量有缓存**：修改 `.env` 后要重启 Jupyter 内核才会生效。
3. **上传是异步的**：追踪数据在后台上传。普通 `.py` 脚本运行很快就结束时，可以在最后调用 `wait_for_all_tracers()`，确保数据上传完成。
4. **SDK 接口在变化**：langsmith 0.14 起，查询追踪记录要用异步的 `client.runs.query()`，创建反馈要提供 `session_id`。遇到 DeprecationWarning 时，按警告中的链接查看新写法。
5. **评估会产生额外的记录**：`evaluate()` 中评估器（如 `llm_judge`）调用模型的记录，默认会放在一个名为 `evaluators` 的项目中。

**总结**

| 功能 | 做什么 | 怎么用 |
|---|---|---|
| Tracing | 记录每次调用的输入输出、耗时、token、费用 | 设置环境变量后自动追踪；自己的函数用 `@traceable`，openai SDK 用 `wrap_openai` |
| 查询追踪数据 | 用代码统计分析 | `await client.aread_project()` + `async for ... in client.runs.query()` |
| Feedback | 给调用打分 | `client.create_feedback(run_id=..., key=..., score=..., session_id=...)` |
| Datasets & Experiments | 用数据对比不同版本的效果 | `create_dataset` + `create_examples` + `client.evaluate()` |
| Evaluators | 自动打分 | 规则函数，或让大模型当裁判 |
| Monitoring 等 | 监控、告警、自动化、人工标注、提示词管理 | 在网页上操作 |